In [1]:
import numpy as np
import tensorflow as tf
from keras.datasets import cifar10
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from art.attacks.evasion import SaliencyMapMethod
from art.estimators.classification  import KerasClassifier
from sklearn.metrics import accuracy_score

In [2]:
tf.compat.v1.disable_eager_execution()

In [3]:
import tensorflow as tf
import json
# download mnist data and split into train and test sets
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
# reshape data to fit model
X_train, X_test = X_train/255, X_test/255
# one-hot encode target column
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)
# create model
model = Sequential()
model.add(Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3), activation='relu'))
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

In [4]:
# compile model using accuracy as a measure of model performance
model.compile(optimizer='adam', loss='categorical_crossentropy',
              metrics=['accuracy'])

# train model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5)

Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - ETA: 0s - loss: 1.4771 - accuracy: 0.4605

C:\Users\nidhi.jain1\Anaconda3\lib\site-packages\keras\engine\training_v1.py:2333: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates = self.state_updates


50000/50000 [==============================] - 234s 5ms/sample - loss: 1.4771 - accuracy: 0.4605 - val_loss: 1.1403 - val_accuracy: 0.5922
Epoch 2/5
50000/50000 [==============================] - 273s 5ms/sample - loss: 1.0999 - accuracy: 0.6120 - val_loss: 0.9239 - val_accuracy: 0.6820
Epoch 3/5
50000/50000 [==============================] - 272s 5ms/sample - loss: 0.9340 - accuracy: 0.6716 - val_loss: 0.8108 - val_accuracy: 0.7171
Epoch 4/5
50000/50000 [==============================] - 180s 4ms/sample - loss: 0.8450 - accuracy: 0.7024 - val_loss: 0.7371 - val_accuracy: 0.7412
Epoch 5/5
50000/50000 [==============================] - 155s 3ms/sample - loss: 0.7809 - accuracy: 0.7266 - val_loss: 0.7851 - val_accuracy: 0.7213


In [5]:
#Create a KerasClassifier for the model
classifier = KerasClassifier(model=model, clip_values=(0, 1),  use_logits=False)
# Generate adversarial examples
x_test = X_test # your test data
y = y_test # the true labels for the test data
jsma = SaliencyMapMethod(classifier=classifier, theta=0.5, gamma=0.5)
#x_test_adv = jsma.generate(x=x_test, y=y)
x_test_adv = jsma.generate(x_test)

C:\Users\nidhi.jain1\Anaconda3\lib\site-packages\keras\engine\training_v1.py:2357: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


JSMA:   0%|          | 0/10000 [00:00<?, ?it/s]

In [6]:
# Evaluate the model's accuracy before adv attack on the test dataset
score1 = model.evaluate(x_test, y_test, verbose=0)
print('Test accuracy:', score1[1])

# Evaluate the model's accuracy on the test dataset
model.fit(X_train, y_train, batch_size=32, epochs=5, validation_data=(x_test_adv, y_test))
score1 = model.evaluate(x_test_adv, y_test, verbose=0)
print('Test accuracy after adversarial attack:', score1[1])

#x_test_adv
#x_test
#print(f"X_train shape: {x_test_adv.shape}")
#print(f"y_train shape: {x_test.shape}")

Test accuracy: 0.7213
Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - 92s 2ms/sample - loss: 0.7335 - accuracy: 0.7437 - val_loss: 1.5976 - val_accuracy: 0.4320
Epoch 2/5
50000/50000 [==============================] - 93s 2ms/sample - loss: 0.7015 - accuracy: 0.7555 - val_loss: 1.7577 - val_accuracy: 0.4100
Epoch 3/5
50000/50000 [==============================] - 92s 2ms/sample - loss: 0.6660 - accuracy: 0.7669 - val_loss: 1.5754 - val_accuracy: 0.4547
Epoch 4/5
50000/50000 [==============================] - 92s 2ms/sample - loss: 0.6388 - accuracy: 0.7785 - val_loss: 1.8848 - val_accuracy: 0.3987
Epoch 5/5
50000/50000 [==============================] - 92s 2ms/sample - loss: 0.6148 - accuracy: 0.7860 - val_loss: 1.7996 - val_accuracy: 0.4087
Test accuracy after adversarial attack: 0.4087
X_train shape: (10000, 32, 32, 3)
y_train shape: (10000, 32, 32, 3)
